In [1]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn import set_config
set_config(transform_output="pandas")

sys.path.append(os.path.abspath('../src'))
from sklearn.model_selection import train_test_split
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from Transformer import Transformer
from InteractionsTransformer import InteractionsTransformer
from PreProcessor import PreProcessor
from ModelCollection import ModelCollection
from PipelineBuilder import PipelineBuilder
from CrossValidation import CrossValidation
from EnsembleBuilder import EnsembleBuilder

In [2]:
config_path = "../src/config.jsonc"
path_train, path_test = "../data/train.csv", "../data/test.csv"
data_loader = DataLoader(path_train, path_test)
train, test = data_loader.load()

In [3]:
test

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
1455,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
1456,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
1457,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal


In [4]:
train['SalePrice'] = np.log1p(train['SalePrice'])
data_splitter = DataSplitter("SalePrice", test_size=0.2)
X_train, X_test, y_train, y_test = data_splitter.split(train)
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_train.shape[0]/(X_train.shape[0]+X_test.shape[0])

((1168, 80), (292, 80), (1168,), (292,), 0.8)

In [5]:
# ML pipeline
pipeline_builder = PipelineBuilder(config_path=config_path, df=train)

$\textbf{OLS}$

In [6]:
## OLS fit on X_train, y_train:

pipeline = pipeline_builder.build(model_name='OLS')
pipeline.fit(X_train, y_train)
preds = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()

           y_test        pred       Diff%
count  292.000000  292.000000  292.000000
mean    11.997654   11.465753    4.499059
std      0.432727    0.539369    2.544203
min     10.471978   10.000000    0.007482
25%     11.751950   11.000000    2.075960
50%     11.945687   11.000000    4.951138
75%     12.250929   12.000000    6.778059
max     13.534474   13.000000   11.251881


In [7]:
## Cross-validation score

n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.13763584697181847), 'std_rmse': np.float64(0.03202011720034662)}


In [8]:
## Hyper-parameter tunning
param_grid = {
    "model__fit_intercept": [False, True]
}
search_ols = cv.hyper_param_tune(X, y, param_grid)

In [9]:
## Predicting on test set 
X_train, y_train = train.drop(columns=['SalePrice']), train['SalePrice']
pipeline.fit(X_train, y_train)
preds = np.expm1(pipeline.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_ols.csv", index=False)

$\textbf{Linear Regression (Regularized Models) - Ridge and Lasso}$

$Ridge:$

In [10]:
pipeline = pipeline_builder.build(model_name='Ridge')
pipeline.fit(X_train, y_train)
preds = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

           y_test        pred       Diff%
count  292.000000  292.000000  292.000000
mean    11.997654   11.469178    4.471380
std      0.432727    0.533170    2.517552
min     10.471978   10.000000    0.007482
25%     11.751950   11.000000    2.075960
50%     11.945687   11.000000    4.901415
75%     12.250929   12.000000    6.764743
max     13.534474   13.000000    9.294103


In [11]:
## Cross-validation score
n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.13449927597738337), 'std_rmse': np.float64(0.031027366366467867)}


In [12]:
## Hyper-parameter tunning
param_grid = {
    "model__alpha": [0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.5, 2.0, 3, 5, 10, 20, 30, 40, 50, 100]
}
search_ridge = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_ridge.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results[['param_model__alpha', 'mean_test_score', 'std_test_score']]

,param_model__alpha,mean_test_score,std_test_score
8,3.0,-0.134393,0.031071
9,5.0,-0.134407,0.031422
7,2.0,-0.134417,0.030952
6,1.5,-0.134444,0.030948
5,1.1,-0.134484,0.031001
4,0.9,-0.134519,0.031060
3,0.7,-0.134582,0.031152
10,10.0,-0.134598,0.032157
2,0.5,-0.134709,0.031289
1,0.3,-0.135022,0.031493


In [13]:
coefs_values = search_ridge.best_estimator_.named_steps["model"].coef_
feature_names = search_ridge.best_estimator_.named_steps["interactions"].feature_names_out_
coefs_df = pd.DataFrame({'Feature': feature_names, "Coefficient": coefs_values})
coefs_df = coefs_df.sort_values(by='Coefficient', key=abs, ascending=False)
coefs_df.head(50)
coefs_df[coefs_df["Feature"] == "GrLivArea_OverallQual"]

,Feature,Coefficient


In [14]:
preds = np.expm1(search_ridge.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_ridge.csv", index=False)

$Lasso:$

In [15]:
pipeline = pipeline_builder.build(model_name='Lasso')
pipeline.fit(X_train, y_train)
preds = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

           y_test   pred       Diff%
count  292.000000  292.0  292.000000
mean    11.997654   12.0    2.780247
std      0.432727    0.0    2.309270
min     10.471978   12.0    0.007482
25%     11.751950   12.0    1.090029
50%     11.945687   12.0    2.110711
75%     12.250929   12.0    3.757012
max     13.534474   12.0   14.591530


In [16]:
## Cross-validation score
n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.3987817488470154), 'std_rmse': np.float64(0.025132411941850535)}


In [17]:
## Hyper-parameter tunning
param_grid = {
    "model__alpha": [0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.5, 2.0, 3, 5, 10, 20, 30, 40, 50, 100]
}
search_lasso = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_lasso.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results[['param_model__alpha', 'mean_test_score', 'std_test_score']]

,param_model__alpha,mean_test_score,std_test_score
0,0.1,-0.213321,0.014709
1,0.3,-0.377950,0.029749
2,0.5,-0.398782,0.025132
3,0.7,-0.398782,0.025132
4,0.9,-0.398782,0.025132
5,1.1,-0.398782,0.025132
6,1.5,-0.398782,0.025132
7,2.0,-0.398782,0.025132
8,3.0,-0.398782,0.025132
9,5.0,-0.398782,0.025132


In [18]:
coefs_values = search_lasso.best_estimator_.named_steps["model"].coef_
feature_names = search_lasso.best_estimator_.named_steps["interactions"].feature_names_out_
coefs_df = pd.DataFrame({'Feature': feature_names, "Coefficient": coefs_values})
coefs_df = coefs_df.sort_values(by='Coefficient', key=abs, ascending=False)
coefs_df.head(10)

,Feature,Coefficient
1,num__OverallQual,0.140672
2,num__GrLivArea,0.055940
21,num__Baths,0.031950
17,num__GarageCars,0.030097
25,num__1stFlrSF,0.018868
211,ord_2__KitchenQual,0.010862
0,num__YrSold,-0.000000
7,num__BedroomAbvGr,0.000000
8,num__LotFrontage,0.000000
5,num__TotRmsAbvGrd,0.000000


In [19]:
preds = np.expm1(search_lasso.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_lasso.csv", index=False)

$\textbf{Random Forest}$

In [20]:
pipeline = pipeline_builder.build(model_name='Random Forest')
pipeline.fit(X_train, y_train)
preds = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

           y_test        pred       Diff%
count  292.000000  292.000000  292.000000
mean    11.997654   11.458904    4.506034
std      0.432727    0.532474    2.485428
min     10.471978   10.000000    0.007482
25%     11.751950   11.000000    2.206949
50%     11.945687   11.000000    5.032615
75%     12.250929   12.000000    6.819481
max     13.534474   13.000000    8.709334


In [21]:
## Cross-validation score
n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.14525885822833834), 'std_rmse': np.float64(0.019104085111615664)}


In [22]:
"""
## Hyper-parameter tunning
param_grid = {
    "model__n_estimators": [500, 1000],
    "model__max_depth": [10, 20],
    "model__min_samples_leaf": [5, 8, 10],
    "model__max_features": [0.3, 0.5, 0.7],
}
search_rf = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_rf.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results.iloc[0]"""

'\n## Hyper-parameter tunning\nparam_grid = {\n    "model__n_estimators": [500, 1000],\n    "model__max_depth": [10, 20],\n    "model__min_samples_leaf": [5, 8, 10],\n    "model__max_features": [0.3, 0.5, 0.7],\n}\nsearch_rf = cv.hyper_param_tune(X, y, param_grid)\nresults = pd.DataFrame(search_rf.cv_results_)\nresults = results.sort_values("mean_test_score", ascending=False)\nresults.iloc[0]'

In [23]:
"""
feature_imp = search_rf.best_estimator_.named_steps["model"].feature_importances_
feature_names = search_rf.best_estimator_.named_steps["preprocess"].get_feature_names_out()
features_df = pd.DataFrame({'Feature': feature_names, "Importance": feature_imp})
features_df = features_df.sort_values(by='Importance', key=abs, ascending=False)
features_df.head(20)"""

'\nfeature_imp = search_rf.best_estimator_.named_steps["model"].feature_importances_\nfeature_names = search_rf.best_estimator_.named_steps["preprocess"].get_feature_names_out()\nfeatures_df = pd.DataFrame({\'Feature\': feature_names, "Importance": feature_imp})\nfeatures_df = features_df.sort_values(by=\'Importance\', key=abs, ascending=False)\nfeatures_df.head(20)'

In [24]:
"""preds = np.expm1(search_rf.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_rf.csv", index=False)"""

'preds = np.expm1(search_rf.best_estimator_.predict(test))\nsubmission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})\nsubmission.to_csv("../data/submission_rf.csv", index=False)'

$\textbf{Gradient Boosting}$

In [25]:
pipeline = pipeline_builder.build(model_name='Gradient Boosting')
pipeline.fit(X_train, y_train)
preds = pd.Series(pipeline.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000788 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 78
[LightGBM] [Info] Start training from score 12.024057
           y_test        pred       Diff%
count  292.000000  292.000000  292.000000
mean    11.997654   11.462329    4.493440
std      0.432727    0.557934    2.496750
min     10.471978   10.000000    0.007482
25%     11.751950   11.000000    2.164631
50%     11.945687   11.000000    4.993554
75%     12.250929   12.000000    6.778059
max     13.534474   13.000000    9.108398


In [26]:
## Cross-validation score
n_folds = 5
cv = CrossValidation(n_folds, pipeline)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.1314492245870276), 'std_rmse': np.float64(0.014460680395474771)}


In [27]:
## Hyper-parameter tunning
param_grid = {
    "model__num_leaves": [15, 31, 63],
    "model__learning_rate": [0.01, 0.03, 0.05],
    "model__min_child_samples": [5, 10, 20],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0]
}
search_gb = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_gb.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results.iloc[0]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003940 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 78
[LightGBM] [Info] Start training from score 12.024057


mean_fit_time                                                              1.711796
std_fit_time                                                                0.04208
mean_score_time                                                            0.096526
std_score_time                                                             0.010611
param_model__colsample_bytree                                                   0.6
param_model__learning_rate                                                     0.05
param_model__min_child_samples                                                   20
param_model__num_leaves                                                          31
param_model__subsample                                                          0.8
params                            {'model__colsample_bytree': 0.6, 'model__learn...
split0_test_score                                                          -0.13611
split1_test_score                                                         -0

In [28]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__colsample_bytree,param_model__learning_rate,param_model__min_child_samples,param_model__num_leaves,param_model__subsample,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
76,1.711796,0.042080,0.096526,0.010611,0.6,0.05,20,31,0.8,"{'model__colsample_bytree': 0.6, 'model__learn...",-0.136110,-0.116538,-0.143939,-0.127114,-0.112185,-0.127177,0.011822,1
75,1.715298,0.118921,0.100709,0.009673,0.6,0.05,20,31,0.6,"{'model__colsample_bytree': 0.6, 'model__learn...",-0.136110,-0.116538,-0.143939,-0.127114,-0.112185,-0.127177,0.011822,1
77,1.654803,0.166273,0.105704,0.018095,0.6,0.05,20,31,1.0,"{'model__colsample_bytree': 0.6, 'model__learn...",-0.136110,-0.116538,-0.143939,-0.127114,-0.112185,-0.127177,0.011822,1
73,1.028681,0.092447,0.095608,0.007913,0.6,0.05,20,15,0.8,"{'model__colsample_bytree': 0.6, 'model__learn...",-0.132607,-0.115273,-0.144350,-0.129990,-0.113905,-0.127225,0.011403,4
74,1.109528,0.086074,0.100756,0.009523,0.6,0.05,20,15,1.0,"{'model__colsample_bytree': 0.6, 'model__learn...",-0.132607,-0.115273,-0.144350,-0.129990,-0.113905,-0.127225,0.011403,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,1.654631,0.050294,0.095026,0.006055,1.0,0.01,5,15,0.8,"{'model__colsample_bytree': 1.0, 'model__learn...",-0.238270,-0.207350,-0.207716,-0.232251,-0.195066,-0.216131,0.016381,238
164,1.254237,0.098690,0.093744,0.007792,1.0,0.01,5,15,1.0,"{'model__colsample_bytree': 1.0, 'model__learn...",-0.238270,-0.207350,-0.207716,-0.232251,-0.195066,-0.216131,0.016381,238
99,0.983270,0.073610,0.096061,0.004189,0.8,0.01,20,15,0.6,"{'model__colsample_bytree': 0.8, 'model__learn...",-0.238781,-0.207570,-0.206805,-0.233747,-0.193807,-0.216142,0.017216,241
100,1.033842,0.091715,0.090894,0.008081,0.8,0.01,20,15,0.8,"{'model__colsample_bytree': 0.8, 'model__learn...",-0.238781,-0.207570,-0.206805,-0.233747,-0.193807,-0.216142,0.017216,241


In [29]:
feature_imp = search_gb.best_estimator_.named_steps["model"].feature_importances_
feature_names = search_gb.best_estimator_.named_steps["preprocess"].get_feature_names_out()
features_df = pd.DataFrame({'Feature': feature_names, "Importance": feature_imp})
features_df = features_df.sort_values(by='Importance', key=abs, ascending=False)
features_df.head(20)

,Feature,Importance
2,num__GrLivArea,175
15,num__BsmtFinSF1,168
37,num__LotArea,165
40,num__GarageYrBlt,158
32,num__SqFeet,157
3,num__TotalBsmtSF,153
25,num__1stFlrSF,138
20,num__OverallCond,124
1,num__OverallQual,114
28,num__GarageArea,99


In [30]:
preds = np.expm1(search_gb.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_gb.csv", index=False)

## Ensemble Models

$\textbf{Voting Regressor}$

In [31]:
model_names = ["Ridge", "Gradient Boosting"]
model_params = {}
weights = [1, 1]
ensemble_builder = EnsembleBuilder(config_path=config_path, df=train)
vot_regressor = ensemble_builder.build_voting(model_names=model_names, model_params=model_params, weights=weights)

vot_regressor.fit(X_train, y_train)
preds = pd.Series(vot_regressor.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001612 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4086
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 80
[LightGBM] [Info] Start training from score 12.024057
           y_test        pred       Diff%
count  292.000000  292.000000  292.000000
mean    11.997654   11.462329    4.493681
std      0.432727    0.539140    2.499061
min     10.471978   10.000000    0.007482
25%     11.751950   11.000000    2.164631
50%     11.945687   11.000000    4.993554
75%     12.250929   12.000000    6.778059
max     13.534474   13.000000    9.108398


In [32]:
## Cross-validation score
n_folds = 5
cv = CrossValidation(n_folds, vot_regressor)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.12370214756170742), 'std_rmse': np.float64(0.021535204000773515)}


In [33]:
## Hyper-parameter (weights) tunning
param_grid = {
    "weights": [
        [1, 1],
        [0, 1],
        [1, 0],
        [0.4, 0.6],
        [0.3, 0.7],
        [0.2, 0.8],
        [0.6, 0.4],
        [0.7, 0.3],
        [0.8, 0.2],
    ]
}
search_vot_reg = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_vot_reg.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results.iloc[0]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000834 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4086
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 80
[LightGBM] [Info] Start training from score 12.024057


mean_fit_time                       1.653942
std_fit_time                        0.163952
mean_score_time                     0.182371
std_score_time                      0.007958
param_weights                     [0.4, 0.6]
params               {'weights': [0.4, 0.6]}
split0_test_score                  -0.121884
split1_test_score                  -0.110225
split2_test_score                  -0.160897
split3_test_score                    -0.1204
split4_test_score                  -0.104415
mean_test_score                    -0.123564
std_test_score                      0.019755
rank_test_score                            1
Name: 3, dtype: object

In [34]:
preds = np.expm1(search_vot_reg.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_vot.csv", index=False)

$\textbf{Stacking Regressor}$

In [35]:
model_names, meta_model_name = ["Ridge", "Gradient Boosting"], "Ridge"
model_params, meta_model_params = {}, {}
stack_regressor = ensemble_builder.build_stacking(model_names=model_names, model_params=model_params, 
                                                  meta_model_name=meta_model_name, meta_model_params=meta_model_params)

stack_regressor.fit(X_train, y_train)
preds = pd.Series(stack_regressor.predict(X_test), index=y_test.index)

results = pd.DataFrame({'y_test': y_test.values, 'pred': preds.values.astype(int)})
results['Diff%'] = 100 * abs((results['y_test']-results['pred']) / results['y_test'])
print(results.describe())

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000469 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4086
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 80
[LightGBM] [Info] Start training from score 12.024057
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001314 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3796
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 80
[LightGBM] [Info] Start training from score 12.021409
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000647 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bi

In [36]:
## Cross-validation score
n_folds = 5
cv = CrossValidation(n_folds, stack_regressor)
X, y = train.drop(columns=['SalePrice']), train['SalePrice']
score_stats = cv.evaluate(X,y)
print(score_stats)

{'mean_rmse': np.float64(0.12503230058271148), 'std_rmse': np.float64(0.023957353205816174)}


In [37]:
## Hyper-parameter (final estimator) tunning
param_grid = {
    "final_estimator__alpha": [0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 4, 6, 10]
}
search_stack_reg = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_stack_reg.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results.iloc[0]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000632 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4086
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 80
[LightGBM] [Info] Start training from score 12.024057
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000423 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3796
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 80
[LightGBM] [Info] Start training from score 12.021409
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000770 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bi

mean_fit_time                                        9.758325
std_fit_time                                         0.232038
mean_score_time                                      0.169896
std_score_time                                        0.00757
param_final_estimator__alpha                              6.0
params                          {'final_estimator__alpha': 6}
split0_test_score                                   -0.121017
split1_test_score                                   -0.110605
split2_test_score                                   -0.167874
split3_test_score                                   -0.119701
split4_test_score                                   -0.103623
mean_test_score                                     -0.124564
std_test_score                                       0.022563
rank_test_score                                             1
Name: 8, dtype: object

In [38]:
preds = np.expm1(search_stack_reg.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../data/submission_stack.csv", index=False)